# Cboe SOXX/SOXL 期权成交量来源探针

假设：上个完整交易日的 SOXX/SOXL 看涨与看跌量构成可能携带不同于价格和波动率的信息。本卡仅检验 Cboe 官方历史下载是否提供该字段；不读取市场结果、拟合或回测。四个请求和预算见 `plan.json`，原字节及收据见 `data/snapshots/` 与 `retrieval_receipts.json`。

In [ ]:
from pathlib import Path
import csv, hashlib, io, json
root = Path.cwd().resolve()
while not (root / "SPEC.md").is_file():
    if root.parent == root:
        raise RuntimeError("workspace root not found")
    root = root.parent
card = root / "research/cboe_soxx_soxl_options_volume_source_20261022"
plan = json.loads((card / "plan.json").read_text())
receipts = json.loads((card / "retrieval_receipts.json").read_text())
assert plan["frozen"] and len(receipts["queries"]) == len(plan["bounded_queries"]) == 4
expected = ["Trade Date", "Underlying", "Product Type", "Exchange", "Volume"]
audit = []
for item in receipts["queries"]:
    assert item["http_status"] == 200 and item["content_type"].startswith("text/csv")
    body = (root / item["relative_path"]).read_bytes()
    assert hashlib.sha256(body).hexdigest() == item["sha256"] and len(body) == item["bytes"]
    reader = csv.DictReader(io.StringIO(body.decode("utf-8-sig")))
    rows = list(reader)
    columns = reader.fieldnames
    assert columns == expected and all(row["Underlying"] == item["symbol"] for row in rows)
    dates = sorted(set(row["Trade Date"] for row in rows))
    exchanges = sorted(set(row["Exchange"] for row in rows))
    values = [int(row["Volume"]) for row in rows]
    assert all(value >= 0 for value in values)
    audit.append({"symbol": item["symbol"], "window": [item["startDate"], item["endDate"]],
                  "rows": len(rows), "dates": len(dates), "first_date": dates[0], "last_date": dates[-1],
                  "exchanges": exchanges, "sum_volume": sum(values), "zero_rows": sum(v == 0 for v in values),
                  "columns": columns, "call_put_split_available": False, "source_sha256": item["sha256"]})
result = {"status": "not_qualified_for_call_put_composition", "plan": "plan.json",
          "reason": "Official responses provide total option volume by date, underlying and Cboe exchange; no call/put or opening/closing fields.",
          "query_audits": audit, "http_gets": 4, "Qlib_D_calendar_reads": 0,
          "models": 0, "accounts": 0, "bootstrap": 0, "market_outcome_reads": 0,
          "alternative_source_observation": "Total Cboe option activity exists for sampled dates, but it is a separate non-directional hypothesis requiring its own frozen source/timing and volume normalization checks.",
          "PIT_qualified": False, "trading_or_promotion_permission": False}
(card / "source_schema_audit.json").write_text(json.dumps(result, ensure_ascii=False, indent=2) + "\n")
summary = {"experiment": plan["experiment"], "status": result["status"],
           "decision": "close_original_call_put_composition_source_probe", "source_schema_audit": "source_schema_audit.json",
           "source_snapshots": [item["relative_path"] for item in receipts["queries"]],
           "recorder_status": "not_started_source_schema_stop", "new_model_fits": 0, "new_accounts": 0,
           "objective_achieved": False, "qualified_forward_days": 0,
           "artifact_policy": plan["artifact_policy"]}
(card / "summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2) + "\n")
(card / "conclusion.md").write_text("# 中文结论\n\nCboe 官方历史下载在四个预定的 SOXX/SOXL 月度样本中返回逐日期、标的、交易所总期权成交量；没有看涨/看跌拆分，也没有开仓/平仓方向。原买卖量构成假设按冻结停止条件关闭，未读取 Qlib 市场日历或未来收益，未拟合或回测。原始响应、URL、获取时间和 SHA-256 均保留。\n\n总期权活跃度是另一个非方向性问题，不能继承本假设的结论；如继续，需要独立冻结、归一化和首次公开时间核查。此卡没有任何 SOXX 超额或交易许可。\n")
print(result["status"], [(x["symbol"], x["window"], x["dates"], x["rows"]) for x in audit])

## 结果

查看同卡 `conclusion.md`。来源未提供买卖权拆分，故没有进入 Qlib 特征、收益标签或账户阶段。